In [183]:
import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np
from functools import reduce
import matplotlib.ticker as mtick
import calendar
from datetime import date
import pathlib
from pathlib import Path
from datetime import datetime, timedelta
import os
from matplotlib.ticker import StrMethodFormatter


import seaborn as sns
import matplotlib.pyplot as plt

In [140]:
#Notes
#Confirm reported electricity use is in kWh before running 
analysis_year = 2025 
timesteps_per_hr = 4
sim_length = 8760 *timesteps_per_hr


In [3]:
def list_folders_pathlib(directory_path):
    """Lists all immediate subdirectories in the given path."""
    p = pathlib.Path(directory_path)
    return [item.name for item in p.iterdir() if item.is_dir()]

In [94]:
def calc_elec_load_profile_spec(dir_path, id):
    target_string = "default_feature_reports"
    file_name = "default_feature_reports.csv"
    elec_load_x = np.full((sim_length, 1), np.nan)

    # Create the DataFrame using the NumPy array and column names
    elec_load = pd.DataFrame(elec_load_x, columns=['test'])
    elec_sum = 0 
    
    for fol in dir_path.iterdir():
        if (fol.is_dir() and id in fol.name):
            for folder in fol.iterdir():
               if folder.is_dir() and target_string in folder.name:
                new_dir_path = os.path.join(dir_path, fol, folder)
                path_to_file = Path(new_dir_path) / file_name
                loads = pd.read_csv(path_to_file)
                elec_inc = loads['Electricity:Facility(kWh)']
                elec_load[id] = elec_inc 
                elec_sum = elec_load.sum(axis=1).sum()
                elec_load = elec_load.drop(columns=['test'])

    return [elec_load, elec_sum] 

In [199]:
def calc_monthly_sums(year, timesteps_per_hr, load_profile):
    start_dt = date(year, 1,1)
    end_dt = date(year+1, 1,1)
    if timesteps_per_hr == 1:
       dates = pd.date_range(start=start_dt, end=end_dt, freq='h')
    elif timesteps_per_hr == 4:
       dates = pd.date_range(start=start_dt, end=end_dt, freq='15min')
    else:
       return "timestep not supported"
 
    dates = dates[:-1] #getting an extra last entry, need to drop
    
    working_df = load_profile.set_index(dates)
    monthly_df = working_df.resample('MS').sum()
    
    
    return working_df, dates, monthly_df

In [219]:
def plot_monthly_elec(id, load_profile): #Note, output aggregates monthly totals under month start date
    buffer = pd.DateOffset(months=1)  
    y_max = load_profile[id].max() * 1.2 
    ax = plt.gca()
    plt.xlim(load_profile[id].index[0] - buffer, load_profile[id].index[-1] + buffer)
    ax.set_xticks(load_profile[id].index[::2])
    plt.grid()
    plt.ylim(0, y_max)
    plt.xlim()
    plt.ylabel('Electricity consumption (kWh)')
    plt.plot(load_profile[id])
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    plt.xticks(rotation=45, ha='right') 

In [77]:
#plot loads and hot water use 
dir_path_test = Path('C:/Users/aallen/Documents/...../createbar_scenario/') #example
id_test = "a20539dd-f643-4532-847a-4fd15663f1bc"
id_test_2 = '01c99d14-837d-487e-944c-d761eb786a5c'

In [207]:
elec_load_01c = calc_elec_load_profile_spec(dir_path_test, id_test_2)[0]
elec_sum_01c = calc_elec_load_profile_spec(dir_path_test, id_test_2)[1]

In [215]:
#create monthly sums 
month_sums_01c = calc_monthly_sums(analysis_year, timesteps_per_hr, elec_load_01c)[2]

In [ ]:
plot_monthly_elec(id_test_2, month_sums_01c)